# 41. CD-OPE-S 구조정의와 초기동등성검증

이 노트북은 4장 실험의 gate입니다. 여기서 통과해야 이후 학습 실험을 진행합니다.

산출물:

- `runs/chapter4_cd_ope_s_design_validity_review.md`
- `runs/manifests/standard/*`
- `runs/cd_ope_s/equivalence_check.json`

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch4_utils.py").exists():
    matches = list(Path.cwd().glob("Deeplearning/*/4장/ch4_utils.py")) + list(Path.cwd().glob("**/ch4_utils.py"))
    if matches:
        NOTEBOOK_DIR = matches[0].parent
    else:
        NOTEBOOK_DIR = Path("Deeplearning") / "Vision 응용" / "4장"
sys.path.insert(0, str(NOTEBOOK_DIR))

from ch4_utils import *

paths = find_ch4_paths()
set_korean_font()
set_seed(41)
paths

Chapter4Paths(chapter4_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장'), chapter3_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장'), runs_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/runs'), manifest_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/runs/manifests'), design_path=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/chapter4_cd_ope_s_architecture_design.md'), validity_path=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/4장/runs/chapter4_cd_ope_s_design_validity_review.md'))

## 41-1. 설계서 타당성 검토 문서 생성

In [2]:
validity_path = build_ch4_design_validity_review()
print(validity_path)
print(validity_path.read_text(encoding="utf-8")[:3000])

C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\4장\runs\chapter4_cd_ope_s_design_validity_review.md
# Chapter 4 CD-OPE-S 실험 설계 타당성 검토

작성일: 2026-05-13

## 1. 결론

현재 설계는 `red` 색상 holdout 붕괴를 검증하기에 타당하다. 3장의 counterfactual 결과가 같은 mask와 형상에서 색상 제거만으로 성능을 회복시켰고, 4장 설계는 이 원인을 stage-1 tokenization에서 직접 검증하도록 되어 있다.

다만 최종 결론은 성능 개선만으로 내려서는 안 된다. `red Dice`, `counterfactual gap`, `stage-1 color separability`, `defect separability`, `gate alpha`가 같은 방향을 가리킬 때만 architecture-level 원인으로 판정해야 한다.

## 2. 강점

- 문제 범위가 `red` color-shift 일반화 하나로 고정되어 있어 실험 질문이 좁고 반증 가능하다.
- CD-OPE-S는 pretrained `Conv_rgb`를 보존하고 residual branch를 zero gate로 추가하므로 from-scratch confounding을 줄인다.
- 성능 지표와 내부 feature 지표를 함께 요구하므로 단순 regularization 효과와 구조 원인 효과를 분리할 수 있다.
- adapter+consistency의 seed variance 문제를 반영해 consistency/style loss ramp-up과 seed-level paired comparison을 포함했다.

## 3. 주요 리스크

| 리스크 | 타당성 영향 | 대응 조건 |
|---|---|---|
| zero-gate 동등성 실패 | pretrained 비교가 깨짐 | `max_abs_logit_diff < 1e-6` 통과 전 학습 금지 |


## 41-2. Chapter 4 manifest 생성

In [3]:
# 최종 실험은 max_per_cell=None을 권장합니다.
# 빠른 smoke run은 2~3으로 줄일 수 있습니다.
MANIFEST_MAX_PER_CELL = None

manifests = create_ch4_manifests(max_per_cell=MANIFEST_MAX_PER_CELL, seed=41)
manifest_summary = pd.DataFrame(
    [{"manifest": k, "path": str(v), "n_rows": len(pd.read_csv(v))} for k, v in manifests.items()]
)
display(manifest_summary)

,manifest,path,n_rows
0,train,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,288
1,eval_matched,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,240
2,eval_stress,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,160
3,eval_matched_probe,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,240
4,eval_color_counterfactual_probe,C:\Users\준승\Desktop\2026-1\Study\Deeplearning\...,192


## 41-3. CD-OPE-S 초기 동등성 검증

In [4]:
equivalence = run_cd_ope_initial_equivalence_check(
    use_global=True,
    use_local=True,
    local_kernel_size=15,
    batch_size=2,
    image_size=128,
    seed=41,
)
equivalence

C:\Users\준승\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 208/208 [00:00<00:00, 13762.23it/s]


[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b0-finetuned-ade-512-512
Key                           | Status   |                                                                                                     
------------------------------+----------+-----------------------------------------------------------------------------------------------------
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 256, 1, 1]) vs model:torch.Size([6, 256, 1, 1])
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([6])                      

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


{'max_abs_logit_diff': 0.0,
 'mean_abs_logit_diff': 0.0,
 'prediction_diff_rate': 0.0,
 'use_global': True,
 'use_local': True,
 'local_kernel_size': 15,
 'has_cd_ope': True,
 'alpha_global': 0.0,
 'alpha_local': 0.0,
 'global_weight_norm': 2.995270252227783,
 'local_weight_norm': 2.995270252227783}

## 41-4. 통과 판정

In [5]:
assert equivalence["max_abs_logit_diff"] < 1e-6, "zero-gate 초기 동등성 실패"
assert equivalence["prediction_diff_rate"] == 0.0, "초기 예측이 vanilla와 다릅니다"
print("초기 동등성 통과: 42번 학습 실험으로 진행할 수 있습니다.")

초기 동등성 통과: 42번 학습 실험으로 진행할 수 있습니다.
